In [1]:
!pip install rank_bm25 sentence-transformers -q


In [3]:
import pandas as pd
import numpy as np
import re
import nltk

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

df = pd.read_csv('/kaggle/input/datasets/b22dckh131hongkhnhvn/arxiv-cs-clean/arxiv_cs_clean.csv')
print(f"Shape: {df.shape}")
print(df[['title','abstract','categories','year']].head(3))

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Shape: (100000, 6)
                                         title  \
0  Modeling Computations in a Semantic Network   
1           A tutorial on conformal prediction   
2       An exploratory study of Google Scholar   

                                            abstract     categories  year  
0  Semantic network research has seen a resurgenc...          cs.AI  2021  
1  Conformal prediction uses past experience to d...  cs.LG stat.ML  2019  
2  The paper discusses and analyzes the scientifi...    cs.DL cs.IR  2019  


In [5]:
STOP_WORDS  = set(stopwords.words('english'))
stemmer     = PorterStemmer()
lemmatizer  = WordNetLemmatizer()

# Các từ thừa đặc thù của paper khoa học — nên bỏ thêm
CUSTOM_STOP = {
    'paper', 'propose', 'proposed', 'method', 'approach',
    'result', 'show', 'use', 'used', 'using', 'based',
    'also', 'two', 'one', 'state', 'art', 'et', 'al',
    'however', 'therefore', 'thus', 'furthermore'
}
STOP_WORDS.update(CUSTOM_STOP)

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'http\S+', '', text)          # bỏ URL
    text = re.sub(r'\$.*?\$', ' math ', text)    # thay LaTeX math → 'math'
    text = re.sub(r'[^a-z\s]', ' ', text)        # bỏ số, ký tự đặc biệt
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text: str) -> list:
    return text.split()

def remove_stopwords(tokens: list) -> list:
    return [t for t in tokens if t not in STOP_WORDS and len(t) > 2]

def stem(tokens: list) -> list:
    return [stemmer.stem(t) for t in tokens]

def lemmatize(tokens: list) -> list:
    return [lemmatizer.lemmatize(t) for t in tokens]

def preprocess_for_tfidf(text: str) -> str:
    tokens = stem(remove_stopwords(tokenize(clean_text(text))))
    return ' '.join(tokens)

def preprocess_for_semantic(text: str) -> str:
    """Pipeline cho TV2 (SBERT) — giữ nguyên câu, chỉ clean nhẹ"""
    # SBERT hiểu ngữ nghĩa tốt hơn khi text còn nguyên cấu trúc câu
    text = text.lower()
    text = re.sub(r'\$.*?\$', ' ', text)   # bỏ LaTeX
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_for_recommend(text: str) -> str:
    """Pipeline cho TV3 (Recommender) — dùng lemmatization"""
    tokens = lemmatize(remove_stopwords(tokenize(clean_text(text))))
    return ' '.join(tokens)

# Test thử
sample = df['abstract'].iloc[0]
print("ORIGINAL:\n", sample[:200])
print("\nTF-IDF:\n",   preprocess_for_tfidf(sample)[:200])
print("\nSEMANTIC:\n", preprocess_for_semantic(sample)[:200])
print("\nRECOMMEND:\n",preprocess_for_recommend(sample)[:200])

ORIGINAL:
 Semantic network research has seen a resurgence from its early history in the cognitive sciences with the inception of the Semantic Web initiative. The Semantic Web effort has brought forth an array o

TF-IDF:
 semant network research seen resurg earli histori cognit scienc incept semant web initi semant web effort brought forth array technolog support encod storag queri semant network data structur world st

SEMANTIC:
 semantic network research has seen a resurgence from its early history in the cognitive sciences with the inception of the semantic web initiative. the semantic web effort has brought forth an array o

RECOMMEND:
 semantic network research seen resurgence early history cognitive science inception semantic web initiative semantic web effort brought forth array technology support encoding storage querying semanti


In [6]:
from tqdm.notebook import tqdm
tqdm.pandas()

print("Đang xử lý cột text cho TF-IDF / BM25...")
df['text_tfidf'] = df['text'].progress_apply(preprocess_for_tfidf)

print("Đang xử lý cột text cho Semantic Search...")
df['text_semantic'] = df['text'].progress_apply(preprocess_for_semantic)

print("Đang xử lý cột text cho Recommender...")
df['text_recommend'] = df['text'].progress_apply(preprocess_for_recommend)

print("\nDone! Xem thử:")
print(df[['title','text_tfidf','text_semantic','text_recommend']].head(2))

Đang xử lý cột text cho TF-IDF / BM25...


  0%|          | 0/100000 [00:00<?, ?it/s]

Đang xử lý cột text cho Semantic Search...


  0%|          | 0/100000 [00:00<?, ?it/s]

Đang xử lý cột text cho Recommender...


  0%|          | 0/100000 [00:00<?, ?it/s]


Done! Xem thử:
                                         title  \
0  Modeling Computations in a Semantic Network   
1           A tutorial on conformal prediction   

                                          text_tfidf  \
0  model comput semant network semant network res...   
1  tutori conform predict conform predict use pas...   

                                       text_semantic  \
0  modeling computations in a semantic network se...   
1  a tutorial on conformal prediction conformal p...   

                                      text_recommend  
0  modeling computation semantic network semantic...  
1  tutorial conformal prediction conformal predic...  


In [7]:
from collections import Counter

# Top từ xuất hiện nhiều nhất sau xử lý
all_words = ' '.join(df['text_tfidf']).split()
word_freq = Counter(all_words)

print(f"Tổng số token:        {len(all_words):,}")
print(f"Vocabulary size:      {len(word_freq):,}")
print(f"\nTop 20 từ phổ biến nhất:")
for word, count in word_freq.most_common(20):
    print(f"  {word:<20} {count:>8,}")

Tổng số token:        10,612,131
Vocabulary size:      67,895

Top 20 từ phổ biến nhất:
  model                 156,245
  learn                 150,729
  network               108,853
  data                   94,466
  imag                   83,456
  train                  77,112
  gener                  74,900
  perform                67,176
  task                   58,530
  neural                 57,821
  algorithm              56,522
  dataset                56,109
  deep                   53,835
  problem                52,278
  result                 49,945
  featur                 49,078
  method                 48,752
  system                 43,279
  predict                41,505
  inform                 38,798


In [8]:
# Lưu file đầy đủ — dùng cho cả nhóm
df.to_csv('/kaggle/working/arxiv_preprocessed.csv', index=False)

# Lưu riêng từng phần cho từng thành viên
df[['id','title','abstract','categories','year','text_tfidf']]\
    .to_csv('/kaggle/working/data_tv1.csv', index=False)

df[['id','title','abstract','categories','year','text_semantic']]\
    .to_csv('/kaggle/working/data_tv2.csv', index=False)

df[['id','title','abstract','categories','year','text_recommend']]\
    .to_csv('/kaggle/working/data_tv3.csv', index=False)

print("Đã lưu xong! Vào tab Output để download.")
print(f"\nKích thước file:")
import os
for f_name in ['arxiv_preprocessed.csv','data_tv1.csv','data_tv2.csv','data_tv3.csv']:
    size = os.path.getsize(f'/kaggle/working/{f_name}') / 1024 / 1024
    print(f"  {f_name:<30} {size:.1f} MB")

Đã lưu xong! Vào tab Output để download.

Kích thước file:
  arxiv_preprocessed.csv         503.5 MB
  data_tv1.csv                   188.5 MB
  data_tv2.csv                   234.1 MB
  data_tv3.csv                   202.3 MB
